# LTV theo cohort và CAC payback: "được chi bao nhiêu để có một khách mới?"

CMO hỏi: **"Một khách mới đáng bao nhiêu tiền, và được chi tối đa bao nhiêu để có một khách?"**. Câu trả lời nhanh
là công thức LTV đơn giản; câu trả lời đáng tin hơn là **LTV theo cohort**: doanh thu luỹ kế mỗi khách theo tuổi,
đọc thẳng từ dữ liệu. Bài này làm cả hai, thấy vì sao chúng lệch nhau, rồi tính **CAC payback** và bảng **CAC tối đa**
theo biên lợi nhuận và thời gian hoàn vốn.

Dữ liệu: Online Retail II (UCI, CC BY 4.0), cả hai sheet, bỏ phần trùng 01-09/12/2010, dòng hợp lệ theo định nghĩa
dùng chung; cohort là tháng mua đầu; tháng cuối 11/2011. **Bộ dữ liệu không có giá vốn và chi phí marketing**: mọi
biên lợi nhuận và CAC trong bài là **ví dụ giả định**, ghi rõ như vậy khi trình bày.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/intermediate/03-cohort-ltv-payback -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Viết sẵn, không cần sửa: đọc Online Retail II (cache parquet) và lọc dòng hợp lệ theo định nghĩa dùng chung
import tempfile
import uuid
from pathlib import Path

CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache trong thư mục tạm, không ghi vào data/raw


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet Online Retail II. Lần đầu đọc xlsx (30-90 giây) rồi cache parquet theo mtime và cỡ tệp."""
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    xlsx = root / "data" / "raw" / "online-retail-ii" / "online_retail_II.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py online-retail-ii")
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet"
    if cache.exists():
        try:
            return pd.read_parquet(cache)
        except (OSError, ValueError):  # cache hỏng: đọc lại
            pass
    tx = pd.read_excel(xlsx, sheet_name=sheet)
    tx = tx.assign(Invoice=tx["Invoice"].astype(str), StockCode=tx["StockCode"].astype(str),
                   Description=tx["Description"].astype("string"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{uuid.uuid4().hex}.tmp")
    tx.to_parquet(tmp, index=False)
    tmp.replace(cache)
    return tx


def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Bỏ hoá đơn huỷ (C...), Quantity <= 0, Price <= 0; thêm revenue = Quantity * Price."""
    s = tx.assign(Invoice=tx["Invoice"].astype(str))
    s = s[~s.Invoice.str.startswith("C") & (s.Quantity > 0) & (s.Price > 0)].copy()
    s["revenue"] = s.Quantity * s.Price
    return s


def cohort_sizes(sales: pd.DataFrame) -> pd.Series:
    """Số khách mới theo tháng mua đầu (cohort), trên dòng có Customer ID."""
    known = sales[sales["Customer ID"].notna()]
    first = known.groupby("Customer ID").InvoiceDate.min().dt.to_period("M")
    return first.value_counts().sort_index().rename_axis("cohort").rename("customers")

## 1. LTV đơn giản và LTV theo cohort

`simple_ltv(aov, orders_per_year, margin, retention) -> float`: lợi nhuận gộp suốt đời của một khách theo công thức

$$LTV = \frac{AOV \times f \times m}{1 - r}$$

với $$f$$ là số đơn mỗi năm, $$m$$ là biên lợi nhuận gộp, $$r$$ là retention năm. `retention` ngoài khoảng
$$[0, 1)$$ thì `ValueError`.

`cohort_ltv(sales, last) -> DataFrame`: bỏ dòng không có `Customer ID`; cohort là tháng mua đầu của khách (trên toàn
bộ `sales`); chỉ dùng doanh thu của tháng `<= last`. Index cohort, cột `k` là **mọi** số nguyên từ 0 tới tuổi của
cohort sớm nhất. Giá trị = **doanh thu luỹ kế** từ tháng 0 tới tháng `k` của cohort **chia số khách của cohort**. Ô
có `k` lớn hơn tuổi cohort (`last - cohort`) là `NaN`.

In [ ]:
def simple_ltv(aov: float, orders_per_year: float, margin: float, retention: float) -> float:
    """AOV × f × m / (1 - r); ValueError nếu r ngoài [0, 1)."""
    # TODO
    raise NotImplementedError


def cohort_ltv(sales: pd.DataFrame, last: pd.Period) -> pd.DataFrame:
    """Doanh thu luỹ kế mỗi khách theo cohort × k; NaN ở ô chưa quan sát được."""
    # TODO: tháng, cohort, k; doanh thu theo (cohort, k) đủ cột; cumsum theo hàng; chia cỡ cohort; che ô k > tuổi
    raise NotImplementedError

## 2. Payback và CAC tối đa

`payback_month(ltv, cac, margin) -> Series`: với mỗi cohort, tháng `k` **đầu tiên** mà lợi nhuận gộp luỹ kế mỗi khách
(`ltv × margin`) **đạt hoặc vượt** `cac`. Chưa đạt trong các tháng đã quan sát thì `<NA>`. Kiểu `Int64`, tên
`payback_month`, index cohort.

`max_cac_table(ltv, sizes, margins, horizons) -> DataFrame`: bảng kịch bản cho tài chính. Dòng là `horizons` (hoàn
vốn trong `h` tháng, tức đọc cột `k = h - 1`), cột là `margins`. Giá trị = `margin ×` LTV trung bình của các cohort đã
quan sát được ở cột đó, **trọng số là cỡ cohort** (`sizes`, index cohort). Tên index `horizon_months`.

In [ ]:
def payback_month(ltv: pd.DataFrame, cac: float, margin: float) -> pd.Series:
    """Tháng k đầu tiên ltv × margin >= cac, theo cohort; Int64, <NA> nếu chưa đạt."""
    # TODO
    raise NotImplementedError


def max_cac_table(ltv: pd.DataFrame, sizes: pd.Series, margins: list[float], horizons: list[int]) -> pd.DataFrame:
    """CAC tối đa để hoàn vốn trong h tháng, theo biên lợi nhuận; trung bình có trọng số cỡ cohort."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu và LTV đơn giản (biên 35% là ví dụ giả định)
LAST = pd.Period("2011-11", "M")
MARGIN = 0.35                                               # ví dụ giả định: dữ liệu không có giá vốn
old, new = read_sheet("Year 2009-2010"), read_sheet("Year 2010-2011")
sales = clean_sales(pd.concat([old[old.InvoiceDate < "2010-12-01"], new], ignore_index=True))
cust = sales[sales["Customer ID"].notna()].assign(month=lambda d: d.InvoiceDate.dt.to_period("M"))
year = cust[(cust.month >= pd.Period("2010-12", "M")) & (cust.month <= LAST)]
prev = cust[(cust.month >= pd.Period("2009-12", "M")) & (cust.month <= pd.Period("2010-11", "M"))]
aov = year.revenue.sum() / year.Invoice.nunique()
orders = year.Invoice.nunique() / year["Customer ID"].nunique()
r = len(set(prev["Customer ID"]) & set(year["Customer ID"])) / prev["Customer ID"].nunique()
print(f"AOV {aov:.0f} GBP, {orders:.2f} đơn/khách/năm, retention năm {r:.1%}")
print(f"LTV đơn giản (lợi nhuận gộp, biên giả định {MARGIN:.0%}): {simple_ltv(aov, orders, MARGIN, r):,.0f} GBP")

In [ ]:
# explore: LTV theo cohort, và khách mới thật sự mang lại bao nhiêu trong 12 tháng đầu
ltv = cohort_ltv(sales, LAST)
sizes = cohort_sizes(sales)
print(ltv.loc["2010-01":"2010-12", [0, 2, 5, 11]].round(0).to_string())

first_month = cust.groupby("Customer ID").month.min()
new_2010 = first_month[(first_month >= pd.Period("2010-01", "M")) & (first_month <= pd.Period("2010-11", "M"))]
c = cust[cust["Customer ID"].isin(new_2010.index)].join(new_2010.rename("cohort"), on="Customer ID")
first_year = c[[(m - k).n <= 11 for m, k in zip(c.month, c.cohort)]].groupby("Customer ID").revenue.sum()
top1 = first_year.nlargest(len(first_year) // 100).sum() / first_year.sum()
print(f"khách mới 01-11/2010: {len(first_year):,}; doanh thu 12 tháng đầu mean {first_year.mean():,.0f}, "
      f"median {first_year.median():,.0f} GBP; 1% khách lớn nhất chiếm {top1:.1%}")
print(f"retention năm của khách cũ {r:.1%} so với m1 của khách mới khoảng 20% (bài Basic 02)")

In [ ]:
# explore: đường LTV theo cohort (2010), đường CAC / biên giả định
fig, ax = plt.subplots(figsize=(9, 4.5))
for cohort in ltv.loc["2010-01":"2010-12"].index:
    row = ltv.loc[cohort].dropna()
    ax.plot(row.index, row.values, lw=1, alpha=0.7, label=str(cohort))
for cac, ls in [(200, "--"), (400, ":")]:
    ax.axhline(cac / MARGIN, color="#e45756", ls=ls, lw=1)
    ax.annotate(f"CAC {cac} / biên {MARGIN:.0%} (giả định)", (0.2, cac / MARGIN + 40), color="#e45756", fontsize=8)
ax.set_xlabel("tháng thứ k sau tháng mua đầu")
ax.set_ylabel("doanh thu luỹ kế mỗi khách (GBP)")
ax.legend(frameon=False, fontsize=7, ncol=2)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("LTV theo cohort 2010: cohort 01/2010 gấp đôi cohort 05/2010 sau 12 tháng", loc="left")
plt.show()

print("payback, CAC 200 (giả định):", payback_month(ltv, 200, MARGIN).loc["2010-01":"2010-12"].to_dict())
print("payback, CAC 400 (giả định):", payback_month(ltv, 400, MARGIN).loc["2010-01":"2010-12"].to_dict())

In [ ]:
# explore: bảng kịch bản CAC tối đa (bỏ cohort 12/2009 vì cắt trái)
table = max_cac_table(ltv.loc[pd.Period("2010-01", "M"):], sizes, margins=[0.25, 0.35, 0.45], horizons=[3, 6, 12])
print("CAC tối đa (GBP) để hoàn vốn trong h tháng, theo biên lợi nhuận gộp giả định:")
print(table.round(0).to_string())

## Đọc số, rồi làm trắc nghiệm

1. LTV đơn giản ra bao nhiêu? Khách mới 2010 thật sự mang lại bao nhiêu doanh thu (mean và median) trong 12 tháng
   đầu? Viết ba lý do công thức đơn giản lạc quan.
2. Cohort nào có LTV 12 tháng cao nhất, thấp nhất? Chênh nhau bao nhiêu lần?
3. Với CAC 200 và 400 GBP (giả định), cohort nào hoàn vốn chậm nhất? Cohort nào chưa hoàn vốn?
4. Đọc bảng kịch bản: nếu tài chính muốn hoàn vốn trong 6 tháng với biên 35%, CAC tối đa là bao nhiêu? Câu nào bạn
   phải nói cùng con số đó?
5. Viết câu trả lời ba dòng cho CMO, có chữ "giả định" ở đúng chỗ.

Xong thì mở app, làm phần **trắc nghiệm** của bài.